### Multi-Model Orchestration: Creating a System to Evaluate AI Responses

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

load_dotenv(override=True)

In [ ]:
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')

gemini_api_url = os.getenv('GEMINI_API_URL')
ollama_api_url = os.getenv('OLLAMA_API_URL')

if anthropic_api_key:
    print(f"Anthropic API key exists and begins with: {anthropic_api_key[:8]}")
else:
    print("Anthropic API key does not exists")

if gemini_api_key:
    print(f"Gemini API key exists ad begins with: {gemini_api_key[:8]}")
else:
    print(f"Gemini API key does not exists")

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
# GET Challenge

anthropic_client = Anthropic(
    api_key=anthropic_api_key
)

response = anthropic_client.messages.create(
    max_tokens=1024,
    messages=messages,
    model="claude-haiku-4-5-20251001"
)
challenge = response.content[0].text
print(challenge)

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": challenge}]

In [ ]:
# ASK ANTHROPIC

model_name = "claude-sonnet-4-6"
response = anthropic_client.messages.create(
    max_tokens=1024,
    messages=messages,
    model=model_name
)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# ASK GEMINI

gemini_client = OpenAI(
    api_key=gemini_api_key,
    base_url=gemini_api_url
)
model_name = "gemini-2.5-flash"

response = gemini_client.chat.completions.create(
    model=model_name,
    messages=messages
)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
ollama_client = OpenAI(
    base_url=ollama_api_url,
    api_key=ollama_api_key
)


In [ ]:
# ASK llma

model_name = "llama3.2:3b"

response = ollama_client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Ask GPT

model_name = "gpt-oss:20b"

response = ollama_client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Note the use of "zip"

for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

In [ ]:
# Note the use of "enumerate"

together = ""

for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{challenge}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

response = anthropic_client.messages.create(
    max_tokens=1024,
    messages=judge_messages,
    model="claude-opus-4-6"
)

results = response.content[0].text
print(results)

In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")